In [1]:
import os, sys

PROJECT_ROOT = "/mnt/WDC10T/tailai_ws/adaptive_compliance_policy/PyriteML"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Added:", PROJECT_ROOT)
print("Can see diffusion_policy:", os.path.isdir(os.path.join(PROJECT_ROOT, "diffusion_policy")))


Added: /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/PyriteML
Can see diffusion_policy: True


In [2]:
import sys
import os
from typing import Dict, Callable, Tuple, List

import numpy as np
import torch
import time
import dill
import hydra
from torch.utils.data import DataLoader

from diffusion_policy.workspace.base_workspace import BaseWorkspace
from diffusion_policy.dataset.base_dataset import BaseImageDataset, BaseDataset
from diffusion_policy.workspace.train_diffusion_unet_image_workspace import TrainDiffusionUnetImageWorkspace

device = torch.device('cuda')   

In [22]:
import os, dill, torch

data_path = "/mnt/WDC10T/tailai_ws/adaptive_compliance_policy/training_outputs"

ckpt_path = os.path.join(
    data_path,
    "2026.02.27_01.24.48_cable_mounting_multimodal_conv_230",
    "checkpoints",
    "epoch=0015-train_loss=0.018.ckpt"   # 或 latest.ckpt
)

assert os.path.isfile(ckpt_path), ckpt_path
print("Loading:", ckpt_path)

Loading: /mnt/WDC10T/tailai_ws/adaptive_compliance_policy/training_outputs/2026.02.27_01.24.48_cable_mounting_multimodal_conv_230/checkpoints/epoch=0015-train_loss=0.018.ckpt


In [23]:
payload = torch.load(open(ckpt_path, "rb"), map_location="cpu", pickle_module=dill)
cfg = payload["cfg"]

cls = hydra.utils.get_class(cfg._target_)
workspace = cls(cfg)
workspace.load_payload(payload, exclude_keys=None, include_keys=None)

policy = workspace.ema_model if cfg.training.use_ema else workspace.model
policy.num_inference_steps = cfg.policy.num_inference_steps

policy.eval().to(device)
policy.reset()

print("Model on:", next(policy.parameters()).device)

key_model_map adding wrench key: robot0_eef_wrench
rgb keys:          ['rgb_0', 'rgb_1', 'rgb_2', 'tactile']
low_dim_keys keys: ['robot0_abs_eef_pos', 'robot0_abs_eef_rot_axis_angle', 'robot0_eef_pos', 'robot0_eef_rot_axis_angle', 'robot0_fingertip_width', 'robot1_abs_eef_pos', 'robot1_abs_eef_rot_axis_angle', 'robot1_eef_pos', 'robot1_eef_rot_axis_angle', 'robot1_fingertip_width']
[CHECK] rgb_2 in encoder forward: torch.Size([1, 2, 3, 224, 224]) torch.float32
==> reduce pretrained obs_encorder's lr
==> rgb keys:  ['rgb_0', 'rgb_1', 'rgb_2', 'tactile']
obs_encorder params: 455
obs_encorder params: 455
Model on: cuda:0


In [24]:
def to_device(x, device):
    if torch.is_tensor(x):
        return x.to(device, non_blocking=True)
    if isinstance(x, dict):
        return {k: to_device(v, device) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return type(x)(to_device(v, device) for v in x)
    return x

In [25]:
from omegaconf import OmegaConf

print("obs_encoder keys:", list(cfg.policy.obs_encoder.keys()))
print("obs_encoder yaml:\n", OmegaConf.to_yaml(cfg.policy.obs_encoder, resolve=True))
print("dataset yaml:\n", OmegaConf.to_yaml(cfg.task.dataset, resolve=True))

obs_encoder keys: ['_target_', 'shape_meta', 'fuse_mode', 'reduce_pretrained_lr', 'vision_encoder_cfg', 'tactile_encoder_cfg', 'force_encoder_cfg']
obs_encoder yaml:
 _target_: diffusion_policy.model.vision.timm_obs_encoder_with_force.TimmObsEncoderWithForce
shape_meta:
  id_list:
  - 0
  - 1
  raw:
    rgb_0:
      shape:
      - 3
      - 224
      - 224
      type: rgb
    rgb_1:
      shape:
      - 3
      - 224
      - 224
      type: rgb
    rgb_2:
      shape:
      - 3
      - 224
      - 224
      type: rgb
    tactile:
      shape:
      - 3
      - 224
      - 224
      type: rgb
    ts_pose_fb_0:
      shape:
      - 7
      type: low_dim
    ts_pose_command_0:
      shape:
      - 7
      type: low_dim
    ts_pose_virtual_target_0:
      shape:
      - 7
      type: low_dim
    stiffness_0:
      shape:
      - 1
      type: low_dim
    wrench_0:
      shape:
      - 6
      type: low_dim
    fingertip_width_0:
      shape:
      - 1
      type: low_dim
    ts_pose_fb_1:
